# Assignment 1 — SFT Baseline (TRL + LoRA)

This notebook trains a **Supervised Fine-Tuning (SFT)** baseline model using **Hugging Face TRL** and **PEFT (LoRA)**.

This version is made for **Google Colab**. It mounts **Google Drive** and saves the model and result files there.

What you get:
- A reproducible SFT run (small model, LoRA adapters)
- Saved model/adapters to disk
- Fixed-prompt outputs before and after SFT, saved to `results/base_samples.json` and `results/sft_samples.json`

Docs:
- TRL main docs: https://huggingface.co/docs/trl/en/index
- SFTTrainer docs: https://huggingface.co/docs/trl/en/sft_trainer

> Recommended small base model: `Qwen/Qwen2.5-0.5B` (or swap to another small base model).


## Step 1: Install packages

Run this cell in Colab first. If Colab asks, restart the runtime after install.


In [ ]:
# Install packages for Colab.
# If Colab says restart runtime, do that and then run the notebook again.
!pip -q install -U "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
                  "trl==0.29.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "wandb" "evaluate"

## Step 2: Mount Google Drive

This step connects Colab to your Google Drive. The trained model and result files will be saved there.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive is ready.')
except ImportError:
    print('This notebook is not running in Google Colab.')


## Step 3: Import and set basic options

This part imports the tools, sets the random seed, chooses the model and dataset, and creates folders in Google Drive.


In [ ]:
import os, json, random
from datetime import datetime
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)

# -----------------------
# Paths
# -----------------------
IN_COLAB = "COLAB_GPU" in os.environ
RUN_NAME = "sft_baseline_lora"
DRIVE_ROOT = "/content/drive/MyDrive/RLHF4LLMs"

if IN_COLAB and not os.path.exists("/content/drive/MyDrive"):
    raise ValueError("Please run the Google Drive mount cell first.")

BASE_DIR = os.path.join(DRIVE_ROOT, RUN_NAME) if IN_COLAB else os.path.abspath(RUN_NAME)
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "final_model")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

for path in [BASE_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)

# -----------------------
# Model choice (small + base model recommended)
# -----------------------
# Base (non-instruct) model is ideal if you want to demonstrate the SFT step clearly.
MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # base
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"  # alternative

# -----------------------
# Dataset choice (SFT instruction data)
# -----------------------
SFT_DATASET = "tatsu-lab/alpaca"

print("Base folder:", BASE_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Final model:", FINAL_MODEL_DIR)
print("Results:", RESULTS_DIR)

## Step 4: Load and format the data

Here we load the Alpaca dataset, keep a smaller sample, turn each example into one text, and split the data into train and test parts.


In [ ]:
# -----------------------
# Load & prepare SFT dataset
# -----------------------
ds = load_dataset(SFT_DATASET, split="train")

# Keep a smaller subset if you're resource-limited (recommended for a first successful run).
MAX_TRAIN_SAMPLES = 5_000
if len(ds) > MAX_TRAIN_SAMPLES:
    ds = ds.shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))

def format_alpaca(example):
    """Convert Alpaca schema -> a single training text.

    Alpaca fields: instruction, input, output.
    We combine them into one text sequence for causal LM training.
    """
    instr = (example.get("instruction") or "").strip()
    inp = (example.get("input") or "").strip()
    out = (example.get("output") or "").strip()

    if inp:
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return {"text": prompt + out}

ds = ds.map(format_alpaca, remove_columns=ds.column_names)

# Train/validation split (for basic monitoring)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

dataset_info = {
    "dataset_name": SFT_DATASET,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "train_rows": len(train_ds),
    "eval_rows": len(eval_ds),
}
with open(os.path.join(RESULTS_DIR, "dataset_info.json"), "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

train_ds, eval_ds

## Step 5: Load tokenizer and base model

This step loads the tokenizer and model. It also sets the pad token and uses memory-saving options.


In [ ]:
# -----------------------
# Tokenizer / Model
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Some base models don't have a pad token; use eos as pad to avoid Trainer issues.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    device_map="auto",
)

# Enable gradient checkpointing to reduce memory (optional; helps on smaller GPUs)
model.gradient_checkpointing_enable()
model.config.use_cache = False

## Step 6: Add LoRA

LoRA trains a small set of extra weights. This makes fine-tuning cheaper and faster.


In [ ]:
# -----------------------
# LoRA config (PEFT)
# -----------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Works for many transformer LMs; if your model errors, inspect module names and adjust.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

## Step 7: Set training options

This part sets the training arguments and builds the SFT trainer. The values below are safe for a small Colab GPU.


In [ ]:
# -----------------------
# SFT training config (TRL)
# -----------------------
# If you have limited VRAM, reduce:
# - per_device_train_batch_size
# - max_length
# and/or increase gradient_accumulation_steps.
sft_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=30,
    logging_steps=20,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    max_length=256,
    packing=False,
    dataset_text_field="text",
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    report_to=[],
    run_name=RUN_NAME,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

## Step 8: Save base model outputs (before SFT)

We use a fixed list of 20 prompts and save base model outputs to `base_samples.json`.


In [ ]:
# -----------------------
# Fixed prompts for before/after comparison
# -----------------------
EVAL_PROMPTS = [
    "Explain supervised learning in simple words.",
    "Explain reinforcement learning in simple words.",
    "What is overfitting? Give one short example.",
    "What is the difference between precision and recall?",
    "Write a polite email to ask for a one-day extension.",
    "Write a short apology message for missing a meeting.",
    "Summarize this topic in three bullet points: climate change.",
    "Given numbers [3, 1, 4, 1, 5], find mean and median.",
    "Sort this list in ascending order: [9, 2, 7, 1].",
    "What is 15% of 260? Show short steps.",
    "Translate to Chinese: 'Machine learning is changing many industries.'",
    "Translate to English: '今天天气很好，我们去公园吧。'",
    "Give two pros and two cons of online learning.",
    "Explain the idea of gradient descent in simple terms.",
    "What is the difference between a dataset and a dataloader?",
    "Write a Python function to check if a number is prime.",
    "Write SQL to find the top 5 highest salaries from table employees.",
    "Give a 4-day beginner workout plan.",
    "How can I study effectively for an exam in one week?",
    "Compare PPO and DPO for alignment in very simple language.",
]

def generate(model, tokenizer, prompt, max_new_tokens=160):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def generate_samples(model, tokenizer, prompts, max_new_tokens=160):
    return [{"prompt": p, "output": generate(model, tokenizer, p, max_new_tokens=max_new_tokens)} for p in prompts]

base_samples = generate_samples(model, tokenizer, EVAL_PROMPTS, max_new_tokens=160)
base_out_path = os.path.join(RESULTS_DIR, "base_samples.json")
with open(base_out_path, "w", encoding="utf-8") as f:
    json.dump(base_samples, f, ensure_ascii=False, indent=2)

print("Base samples saved to:", base_out_path)
base_samples[0]

## Step 9: Train the model

Now the trainer learns from the SFT data.


In [ ]:
# -----------------------
# Train
# -----------------------
train_result = trainer.train()

train_metrics = dict(train_result.metrics)
train_metrics["model_name"] = MODEL_NAME
train_metrics["dataset_name"] = SFT_DATASET
train_metrics["saved_at"] = datetime.now().isoformat()

metrics_path = os.path.join(RESULTS_DIR, "train_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, ensure_ascii=False, indent=2)

train_result

## Step 10: Save the result

This saves the LoRA adapter and the tokenizer to disk.


In [ ]:
# -----------------------
# Save adapters + tokenizer
# -----------------------
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

save_info = {
    "base_dir": BASE_DIR,
    "checkpoint_dir": CHECKPOINT_DIR,
    "final_model_dir": FINAL_MODEL_DIR,
    "results_dir": RESULTS_DIR,
}
with open(os.path.join(RESULTS_DIR, "save_info.json"), "w", encoding="utf-8") as f:
    json.dump(save_info, f, ensure_ascii=False, indent=2)

print("Saved final model to:", FINAL_MODEL_DIR)
print("Saved result files to:", RESULTS_DIR)

## Step 11: Save SFT outputs on the same prompts

We use the same 20 prompts from Step 8 and save post-training outputs to `sft_samples.json`.


In [ ]:
# -----------------------
# Save SFT outputs using the same fixed prompts
# -----------------------
sft_samples = generate_samples(trainer.model, tokenizer, EVAL_PROMPTS, max_new_tokens=160)

sft_out_path = os.path.join(RESULTS_DIR, "sft_samples.json")
with open(sft_out_path, "w", encoding="utf-8") as f:
    json.dump(sft_samples, f, ensure_ascii=False, indent=2)

base_out_path = os.path.join(RESULTS_DIR, "base_samples.json")
if os.path.exists(base_out_path):
    with open(base_out_path, "r", encoding="utf-8") as f:
        base_samples = json.load(f)
    paired_samples = [
        {
            "prompt": EVAL_PROMPTS[i],
            "base_output": base_samples[i]["output"],
            "sft_output": sft_samples[i]["output"],
        }
        for i in range(min(len(base_samples), len(sft_samples), len(EVAL_PROMPTS)))
    ]
    paired_out_path = os.path.join(RESULTS_DIR, "paired_samples.json")
    with open(paired_out_path, "w", encoding="utf-8") as f:
        json.dump(paired_samples, f, ensure_ascii=False, indent=2)
    print("Paired samples saved to:", paired_out_path)

print("SFT samples saved to:", sft_out_path)
sft_samples[0]

## Next steps (for your assignment)

1) **Reward Modeling (RM)**: convert preference datasets into `prompt, chosen, rejected` and train a reward model.  
2) **RLHF**: run DPO (easiest) or PPO using TRL.  
3) **Evaluation**: keep a fixed prompt set, compute win-rate with held-out preference pairs, and do blind human comparisons (baseline vs RLHF).
